In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import joblib
from src.pipeline_components import fillna_text, fillna_cat, LeakSafeTargetEncoder

train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")

In [2]:
NUM_COLS = ["item_condition_id", "shipping", "category_depth", "name_length",
            "desc_length", "name_word_count", "has_description", "is_branded"]
CAT_COLS = ["main_category", "sub_category", "sub_sub_category", "condition_label"]

name_pipe = Pipeline([("fillna", FunctionTransformer(fillna_text)),
                       ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, max_df=0.9, stop_words="english"))])
desc_pipe = Pipeline([("fillna", FunctionTransformer(fillna_text)),
                       ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=2, max_df=0.9, stop_words="english"))])
cat_pipe = Pipeline([("fillna", FunctionTransformer(fillna_cat)),
                      ("ohe", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer(transformers=[
    ("name_tfidf", name_pipe, "name"),
    ("desc_tfidf", desc_pipe, "item_description"),
    ("num_scale", StandardScaler(), NUM_COLS),
    ("cat_ohe", cat_pipe, CAT_COLS),
    ("cat_encode", LeakSafeTargetEncoder(group_col="main_category", smoothing=10), ["main_category", "price"]),
    ("brand_encode", LeakSafeTargetEncoder(group_col="brand_name", smoothing=10), ["brand_name", "price"]),
])

In [3]:
full_pipeline = Pipeline([("preprocess", preprocessor), ("model", Ridge(alpha=5.0, random_state=42))])

y_train = train["log_price"]
y_val = val["log_price"]

full_pipeline.fit(train, y_train)
pred = full_pipeline.predict(val)
rmsle = np.sqrt(mean_squared_error(y_val, pred))
print("Pipeline val RMSLE:", round(rmsle, 4))

Pipeline val RMSLE: 0.5214


In [4]:
joblib.dump(full_pipeline, "../models/mercari_pipeline_final.joblib")
print("Saved.")

Saved.
